In [23]:
using Pkg
project_root = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : abspath(joinpath(pwd(), ".."))
Pkg.activate(project_root)

using QuantumDevices
using QuantumToolbox

import QuantumDevices: DeviceParameter

  Activating project at `~/Documents/CodingProjects/QuantumDevices`


## Setting Up Components and Models

### Multimode Cavity

In [2]:
# Component specs remain symbolic until numerical lowering.
transmon = Component(TransmonSpec(0.25, 20.0; ng = 0.0), :transmon)
mode1 = Component(ResonatorSpec(6.0; dimension = 5), :mode1)
mode2 = Component(ResonatorSpec(7.2; dimension = 4), :mode2)

display(transmon)
display(mode1)
display(mode2)

Component :transmon
  Name: transmon
  Spec: TransmonSpec
  Parameters:
    EC = 0.25 [fixed, required] ∈ positive real
    EJ = 20.0 [fixed, required] ∈ positive real
    dimension = Inf [fixed, required] ∈ any
    ng = 0.0 [fixed, required] ∈ real
  Operators:
    identity
    n
    tunneling
  Hamiltonian: 4 × EC × (n - ng × identity)² - EJ × tunneling / 2
  Metadata: none

Component :mode1
  Name: mode1
  Spec: ResonatorSpec
  Parameters:
    dimension = 5 [fixed, required] ∈ 1..9223372036854775807
    frequency = 6.0 [fixed, required] ∈ positive real
  Operators:
    a
    adag
    n
    number
    … 2 more
  Hamiltonian: frequency × n
  Metadata: none

Component :mode2
  Name: mode2
  Spec: ResonatorSpec
  Parameters:
    dimension = 4 [fixed, required] ∈ 1..9223372036854775807
    frequency = 7.2 [fixed, required] ∈ positive real
  Operators:
    a
    adag
    n
    number
    … 2 more
  Hamiltonian: frequency × n
  Metadata: none

In [3]:
g1 = QuantumDevices.DeviceParameter(:g1; default = 0.025)
g2 = QuantumDevices.DeviceParameter(:g2; default = 0.018)

transmon_mode1 = InteractionSpec(
    :transmon_mode1,
    param(:g1) * op(:transmon, :n) * op(:mode1, :q),
)
transmon_mode2 = InteractionSpec(
    :transmon_mode2,
    param(:g2) * op(:transmon, :n) * op(:mode2, :q),
)

multimode_cavity_spec = ModelSpec(
    :multimode_cavity;
    components = Dict(:transmon => transmon, :mode1 => mode1, :mode2 => mode2),
    interactions = Dict(
        :transmon_mode1 => transmon_mode1,
        :transmon_mode2 => transmon_mode2,
    ),
    dims = Dict(:transmon => 6, :mode1 => 5, :mode2 => 4),
    initialization_dims = Dict(:transmon => 121),
    parameters = Dict(:g1 => g1, :g2 => g2),
)

multimode_cavity = model(multimode_cavity_spec)

QuantumDeviceModel :multimode_cavity
  Basis: product
  Hamiltonian: 120×120
  Dressed states: 120 labeled states
  Energies: 120 labeled values
  State order: mode1, mode2, transmon
  State labels: 120 product labels
  Minimum tracked overlap: 0.996211537222355
  Operators:
    mode1.a [120×120]
    mode1.adag [120×120]
    mode1.n [120×120]
    mode1.number [120×120]
    … 12 more
  Resolved parameters: g1=0.025, g2=0.018
  Provenance:
    ModelSpec :multimode_cavity [dims=mode1=5,mode2=4,transmon=6; dimension=120] (3 components, 2 interactions)
      components: mode1[5], mode2[4], transmon[6]
      initialization: transmon=121
      parameters: g1=0.025, g2=0.018
      interactions: transmon_mode1: g1 × transmon.n × mode1.q; transmon_mode2: g2 × transmon.n × mode2.q
      └─ ModelSpec :transmon [dims=transmon=121; dimension=121] (1 component, 0 interactions)
         components: transmon[121]
         initialization: transmon=121

In [4]:
reference_label = (0, 0, 0)  # positions follow multimode_cavity.state_order

(
    full_hamiltonian_size = size(multimode_cavity.hamiltonian),
    dressed_state_count = length(multimode_cavity.states),
    dressed_state = multimode_cavity.states[reference_label],
    dressed_energy = multimode_cavity.energies[reference_label],
    state_order = multimode_cavity.state_order,
    selected_labels = multimode_cavity.state_labels,
    minimum_overlap = minimum(multimode_cavity.state_overlaps),
    provenance = multimode_cavity.spec.children,
)

# Any model node can become a normal frozen Component for another parent.
multimode_component = component(multimode_cavity)
numerical(multimode_component, op(:multimode_cavity, :mode1, :a))


Quantum Object:   type=Operator()   dims=([120], [120])   size=(120, 120)   ishermitian=false
120×120 Matrix{ComplexF64}:
          0.0+0.0im      0.926113+0.0im  …           0.0+0.0im
   0.00097451+0.0im           0.0+0.0im     -1.92562e-18+0.0im
   0.00235755+0.0im           0.0+0.0im     -5.02395e-16+0.0im
  -4.15549e-5+0.0im           0.0+0.0im     -4.64491e-16+0.0im
          0.0+0.0im     0.0018001+0.0im              0.0+0.0im
          0.0+0.0im    0.00080161+0.0im  …           0.0+0.0im
          0.0+0.0im    0.00189027+0.0im              0.0+0.0im
          0.0+0.0im   -3.86361e-5+0.0im              0.0+0.0im
          0.0+0.0im    6.81725e-6+0.0im              0.0+0.0im
          0.0+0.0im   -6.21144e-8+0.0im              0.0+0.0im
             ⋮                           ⋱  
          0.0+0.0im  -5.73491e-15+0.0im              0.0+0.0im
          0.0+0.0im  -1.88175e-15+0.0im              0.0+0.0im
          0.0+0.0im  -4.74241e-16+0.0im              0.0+0.0im
          0.0

In [5]:
# Compact mathematical expressions and structured summaries are used automatically.
display(transmon_mode1)
display(transmon_mode2)
display(multimode_cavity.spec.children[:transmon])
display(multimode_cavity)
display(multimode_component)

InteractionSpec :transmon_mode1
  Expression: g1 × transmon.n × mode1.q
  Components: mode1, transmon
  Metadata: none

InteractionSpec :transmon_mode2
  Expression: g2 × transmon.n × mode2.q
  Components: mode2, transmon
  Metadata: none

ModelSpec :transmon [dims=transmon=121; dimension=121] (1 component, 0 interactions)
  components: transmon[121]
  initialization: transmon=121

QuantumDeviceModel :multimode_cavity
  Basis: product
  Hamiltonian: 120×120
  Dressed states: 120 labeled states
  Energies: 120 labeled values
  State order: mode1, mode2, transmon
  State labels: 120 product labels
  Minimum tracked overlap: 0.996211537222355
  Operators:
    mode1.a [120×120]
    mode1.adag [120×120]
    mode1.n [120×120]
    mode1.number [120×120]
    … 12 more
  Resolved parameters: g1=0.025, g2=0.018
  Provenance:
    ModelSpec :multimode_cavity [dims=mode1=5,mode2=4,transmon=6; dimension=120] (3 components, 2 interactions)
      components: mode1[5], mode2[4], transmon[6]
      initialization: transmon=121
      parameters: g1=0.025, g2=0.018
      interactions: transmon_mode1: g1 × transmon.n × mode1.q; transmon_mode2: g2 × transmon.n × mode2.q
      └─ ModelSpec :transmon [dims=transmon=121; dimension=121] (1 component, 0 interactions)
         components: transmon[121]
         initialization: transmon=121

Component :multimode_cavity
  Name: multimode_cavity
  Spec: FrozenModelSpec
  Parameters:
    dimension = 120 [fixed, required] ∈ 120..120
  Operators:
    mode1.a
    mode1.adag
    mode1.n
    mode1.number
    … 13 more
  Hamiltonian: hamiltonian
  Metadata: 2 keys: resolved_parameters, source_model

### Flux Tunable Transmon Array

In [7]:
qubit1 = Component(FluxTunableTransmonSpec(0.25, 20.0), :qubit1)
coupler = Component(FluxTunableTransmonSpec(0.25, 21.0), :coupler)
qubit2 = Component(FluxTunableTransmonSpec(0.25, 22.0), :qubit2)

g1c = QuantumDevices.DeviceParameter(:g1; default = 0.025)
g2c = QuantumDevices.DeviceParameter(:g2; default = 0.028)
g12 = QuantumDevices.DeviceParameter(:g2; default = 0.001)


interaction1C = InteractionSpec(
    :interaction1C,
    param(:g1c) * op(:qubit1, :n) * op(:coupler, :n),
)

interaction2C = InteractionSpec(
    :interaction2C,
    param(:g2c) * op(:qubit2, :n) * op(:coupler, :n),
)

interaction12 = InteractionSpec(
    :interaction12,
    param(:g12) * op(:qubit2, :n) * op(:qubit2, :n),
)

tunable_coupler_spec = ModelSpec(
    :tunable_coupler;
    components = Dict(:qubit1 => qubit1, :coupler => coupler, :qubit2 => qubit2),
    interactions = Dict(
        :interaction1C => interaction1C,
        :interaction2C => interaction2C,
        :interaction12 => interaction12
    ),
    dims = Dict(:qubit1 => 6, :coupler => 6, :qubit2 => 6),
    initialization_dims = Dict(:qubit1 => 121, :coupler => 121, :qubit2 => 121),
    parameters = Dict(:g1c => g1c, :g2c => g2c, :g12 => g12),
)

tunable_coupler = model(tunable_coupler_spec)

┌ Warning: Dressed-state tracking overlap below 0.5 at step 20 for tracked states [126].
└ @ QuantumDevices /Users/gavinrockwood/Documents/CodingProjects/QuantumDevices/src/utils/state_tracking.jl:357


QuantumDeviceModel :tunable_coupler
  Basis: product
  Hamiltonian: 216×216
  Dressed states: 216 labeled states
  Energies: 216 labeled values
  State order: coupler, qubit1, qubit2
  State labels: 216 product labels
  Minimum tracked overlap: 0.49705179012828554
  Operators:
    coupler.hamiltonian [216×216]
    coupler.identity [216×216]
    coupler.n [216×216]
    coupler.tunneling [216×216]
    … 8 more
  Resolved parameters: g12=0.001, g1c=0.025, g2c=0.028
  Provenance:
    ModelSpec :tunable_coupler [dims=coupler=6,qubit1=6,qubit2=6; dimension=216] (3 components, 3 interactions)
      components: coupler[6], qubit1[6], qubit2[6]
      initialization: coupler=121, qubit1=121, qubit2=121
      parameters: g12=0.001, g1c=0.025, g2c=0.028
      interactions: interaction12: g12 × qubit2.n × qubit2.n; interaction1C: g1c × qubit1.n × coupler.n; interaction2C: g2c × qubit2.n × coupler.n
      ├─ ModelSpec :coupler [dims=coupler=121; dimension=121] (1 component, 0 interactions)
      │  

In [ ]:
function get_parameters(model::ModelSpec)
    parameters = Dict{String, DeviceParameter}()
    for component in keys(model.components)
        for parameter in keys(model.components[component].parameters)
            parameters[string(component, "/", parameter)] = model.components[component].parameters[parameter]
        end
    end
    for parameter in keys(model.parameters)
        parameters[string("ModelSpec/",parameter)] = model.parameters[parameter]
    end
    return parameters
end

function get_controls(model)

get_parameters (generic function with 2 methods)

In [35]:
get_parameters(tunable_coupler_spec)

Dict{String, DeviceParameter} with 21 entries:
  "qubit1/EJ1"        => Parameter(EJ1=10.0, fixed)
  "ModelSpec/g1c"     => Parameter(g1=0.025, fixed)
  "coupler/dimension" => Parameter(dimension=Inf, fixed)
  "qubit1/EJ2"        => Parameter(EJ2=10.0, fixed)
  "qubit2/dimension"  => Parameter(dimension=Inf, fixed)
  "coupler/EC"        => Parameter(EC=0.25, fixed)
  "qubit2/EJ2"        => Parameter(EJ2=11.0, fixed)
  "coupler/EJ2"       => Parameter(EJ2=10.5, fixed)
  "coupler/flux"      => Parameter(flux=0.0)
  "ModelSpec/g12"     => Parameter(g2=0.001, fixed)
  "qubit1/ng"         => Parameter(ng=0.0, fixed)
  "qubit1/dimension"  => Parameter(dimension=Inf, fixed)
  "qubit2/ng"         => Parameter(ng=0.0, fixed)
  "coupler/ng"        => Parameter(ng=0.0, fixed)
  "coupler/EJ1"       => Parameter(EJ1=10.5, fixed)
  "ModelSpec/g2c"     => Parameter(g2=0.028, fixed)
  "qubit2/EJ1"        => Parameter(EJ1=11.0, fixed)
  "qubit2/flux"       => Parameter(flux=0.0)
  "qubit2/EC"         =

## Defining Gates

In [6]:
struct GateSpec
    schematic
    parameters
    meta
end

## Building a Device

In [7]:
@kwdef struct QuantumDevice
    components::Tuple{Vararg{Component}}
    interactions::Tuple{Vararg{InteractionSpec}}
    schematics::Dict{Symbol, ModelSpec} = Dict()
end


QuantumDevice

In [8]:
transmon = Component(TransmonSpec(0.25, 20.0; ng = 0.0), :transmon)
mode1 = Component(ResonatorSpec(6.0; dimension = 5), :mode1)
mode2 = Component(ResonatorSpec(7.2; dimension = 4), :mode2)

g1 = QuantumDevices.DeviceParameter(:g1; default = 0.025)
g2 = QuantumDevices.DeviceParameter(:g2; default = 0.018)

transmon_mode1 = InteractionSpec(
    :transmon_mode1,
    param(:g1) * op(:transmon, :n) * op(:mode1, :q),
)
transmon_mode2 = InteractionSpec(
    :transmon_mode2,
    param(:g2) * op(:transmon, :n) * op(:mode2, :q),
)

MultimodeCavityDevice = QuantumDevice(components = (transmon, mode1, mode2), interactions = (transmon_mode1, transmon_mode2))

QuantumDevice((Component(:transmon, TransmonSpec; 4 parameters, 3 operators), Component(:mode1, ResonatorSpec; 2 parameters, 6 operators), Component(:mode2, ResonatorSpec; 2 parameters, 6 operators)), (InteractionSpec(:transmon_mode1, g1 × transmon.n × mode1.q), InteractionSpec(:transmon_mode2, g2 × transmon.n × mode2.q)), Dict{Symbol, ModelSpec}())

In [9]:
typeof(transmon_mode1)

InteractionSpec